# Argus — plate detector training (Kaggle GPU)

Trains the single-class Indian license-plate detector. This is the **primary**
training path; the local CPU fallback (`--cpu`) exists only if Kaggle is
unavailable and is 15-25x slower.

## Before you run anything

| Setting (right panel) | Value |
|---|---|
| **Accelerator** | GPU T4 x2 (or P100) |
| **Internet** | **On** — required to `pip install` and clone the repo |
| **Input → Add Data** | an Indian number plate dataset in **YOLO format** |

A new or unverified Kaggle account cannot enable GPU or Internet — both toggles
stay greyed out. Fix it once at kaggle.com/settings → Phone Verification.

Runtime: ~20-40 s/epoch, so 50 epochs is roughly 20-35 minutes.

## Step 1 — Configure

Click the attached dataset in the right panel to see its mount path, and paste
it into `SRC`.

In [ ]:
SRC     = "/kaggle/input/CHANGE-ME"   # <- attached dataset root
EPOCHS  = 50
# 3000 was the first run and produced mAP50 0.928 on held-out Indian plates.
# The dataset has 8823 images; set SUBSET = 8000 to use all of them. Expect
# roughly 1 hour on a T4 for maybe +1 point of mAP50 -- worth it only once
# OCR yield (currently 65%) has stopped being the limiting factor.
SUBSET  = 3000    # train images
VAL     = 400


## Step 2 — Confirm the GPU and install

`lap` is BoT-SORT's assignment solver. Installing it here rather than letting
Ultralytics AutoUpdate it mid-run avoids a pip install firing in the middle of
training.

In [ ]:
!nvidia-smi
!pip -q install ultralytics lap

## Step 3 — Clone the repo

The notebook calls `ml/prepare_dataset.py` and `ml/train_plate.py` from the
repo. **No training code is duplicated here**, so the Kaggle path and the local
path cannot drift apart.

In [ ]:
%cd /kaggle/working
!rm -rf Argus && git clone -q https://github.com/Deeptanshu789/Argus.git
%cd /kaggle/working/Argus

## Step 4 — Get the dataset

No Kaggle "Add Data" and no Roboflow account needed — this pulls a
Roboflow-exported plate dataset straight from Hugging Face over plain HTTPS.

It is a general vehicle-registration-plate set, not Indian-specific. That is
fine for the **detector**: a plate is a high-contrast rectangle on a vehicle and
that geometry transfers. The Indian-specific work lives in *recognition* —
PaddleOCR plus `correct_plate()` in `ml/sidecar.py`, which validates
`XX 00 XX 0000` and the state code. Swap in an Indian dataset later if plate
localisation turns out to be the weak link; nothing else changes.

To use your own dataset instead, replace this cell with
`SRC = "/kaggle/input/your-dataset-slug"`.

In [ ]:
# A dataset that is known to work, needs no account, and downloads in ~1 min.
# 8823 images, Roboflow-exported. Its annotations are COCO JSON, which
# prepare_dataset.py converts -- most public plate datasets are NOT YOLO .txt,
# and finding that out after unzipping 155 MB is how an afternoon disappears.
BASE = "https://huggingface.co/datasets/keremberke/license-plate-object-detection/resolve/main/data"

!mkdir -p /kaggle/working/plates
!cd /kaggle/working/plates && curl -sL -o train.zip {BASE}/train.zip && unzip -q -o train.zip
!cd /kaggle/working/plates && curl -sL -o valid.zip {BASE}/valid.zip && unzip -q -o valid.zip

SRC = "/kaggle/working/plates"
print("SRC =", SRC)
!du -sh {SRC}

## Step 5 — Convert and prepare

`prepare_dataset.py` reads YOLO `.txt`, COCO `.json` or Pascal VOC `.xml` and
always writes YOLO, which is what Ultralytics trains on. It reports which format
it found. Boxes collapse to a single class, `license_plate` — datasets that also
label vehicles are filtered by category name rather than merged, because a
detector trained on "plate OR car, both class 0" finds cars.

In [ ]:
!python ml/prepare_dataset.py --src "{SRC}" --subset {SUBSET} --val {VAL}
!cat datasets/plates/data.yaml

## Step 6 — Train

GPU defaults from `ml/train_plate.py`: `imgsz=640, batch=32, amp=True,
freeze=0`.

Note `freeze=0` — the backbone is **not** frozen. Freezing it is a CPU
concession that costs accuracy, and on a T4 there is no reason to pay it.

In [ ]:
!python ml/train_plate.py --epochs {EPOCHS} --device 0

## Step 7 — Read the score

**Go/no-go bar: `mAP50 >= 0.85`.**

In [ ]:
import csv, pathlib

R = pathlib.Path("runs/detect/plate")
rows = list(csv.DictReader(open(R / "results.csv")))
last = {k.strip(): v for k, v in rows[-1].items()}
m50   = float(last["metrics/mAP50(B)"])
m5095 = float(last["metrics/mAP50-95(B)"])
print(f"epochs run : {len(rows)}")
print(f"mAP50      : {m50:.3f}")
print(f"mAP50-95   : {m5095:.3f}\n")

if m50 >= 0.85:
    print("PASS — export the weights (Step 8).")
elif m50 >= 0.70:
    print("MARGINAL — raise SUBSET toward 15000 and re-run. A GPU run costs minutes.")
else:
    print("FAIL — this is almost certainly a DATA problem, not a training one.")
    print("Single-class, tight-boxed plates reach 0.85 readily; more epochs will")
    print("not fix a bad conversion. Re-check Step 4: labels must be class 0 with")
    print("normalized xywh boxes.")

In [ ]:
from IPython.display import Image, display
display(Image(f"{R}/results.png"))

## Step 8 — Export the weights

Download `argus-plate-weights.zip` from the **Output** panel on the right, then
on the build machine:

```bash
cd ~/code/Argus
mkdir -p runs/detect/plate/weights
unzip ~/Downloads/argus-plate-weights.zip -d runs/detect/plate/weights
./.venv/bin/python ml/export_onnx.py --weights runs/detect/plate/weights/best.pt
```

`export_onnx.py` produces OpenVINO int8. Training was the only GPU step —
inference stays on the laptop CPU, which is why the export matters.

In [ ]:
!cd runs/detect/plate/weights && zip -q /kaggle/working/argus-plate-weights.zip best.pt last.pt
!ls -lh /kaggle/working/argus-plate-weights.zip

## If the session died mid-run

Kaggle kills long sessions. Re-run Steps 2, 3 and 5, then uncomment and run
this — it picks up from the last checkpoint rather than starting over.

In [ ]:
# !python ml/train_plate.py --epochs {EPOCHS} --device 0 --resume

## Done — next steps are on the laptop

```bash
cd ~/code/Argus && git pull
mkdir -p runs/detect/plate/weights
unzip ~/Downloads/argus-plate-weights.zip -d runs/detect/plate/weights
./.venv/bin/python ml/export_onnx.py \
    --weights runs/detect/plate/weights/best.pt --fp32
```

`--fp32` because it needs nothing but the weights, and measures 9 ms/frame on
the build machine against a 50 ms budget. int8 would need the calibration
dataset — which is here on Kaggle, not there — for another 2x on a number that
is already 5x clear.